# Chapter 33 — Testing: pure functions, adapters, GUI smoke tests, and fixtures

**Book role:** Quality assurance  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Write unit tests for analytical and validation functions.
- Use fixtures and fakes for file and database boundaries.
- Test GUI behaviors in small deterministic pieces.
- Build a regression suite around important business metrics.

## 1. Test computations without a GUI

Pure functions are the cheapest place to gain confidence. A filter or KPI service should take a DataFrame and criteria and return deterministic results. Test boundary cases: empty frame, missing columns, null values, duplicate keys, zero denominator, and exact date endpoints. Use a tiny fixture whose expected output is simple enough to verify manually.

A failing analytical test should tell you which business rule changed. Prefer assertions on specific columns and metric values to a snapshot of every row when the entire shape is not important. For floating-point calculations, define a justified tolerance; for financial values, prefer a precise decimal contract and explicit rounding policy.

### Worked example 1: Unit-test an analytical function

Keep the calculation in a normal module so pytest does not need to start Tkinter. The empty/zero case is a deliberate domain decision, not an incidental implementation detail.

In [ ]:
import pandas as pd
import pytest

def profit_margin(revenue: float, profit: float) -> float | None:
    return profit / revenue if revenue else None

def test_profit_margin():
    assert profit_margin(100, 25) == 0.25
    assert profit_margin(0, 0) is None
    assert profit_margin(200, -10) == -0.05

# Save as a test module and run: python -m pytest -q
# The code above defines a pytest test; pytest must be installed to run it.

## 2. Mock external boundaries, not business truth

A fake repository can return a known DataFrame or raise a simulated connection error. This tests controller behavior without requiring a running MySQL/MongoDB service. Integration tests should still run against a controlled test database when the purpose is to verify driver behavior, query syntax, constraints, or transaction semantics. Do not mock away the very thing a test claims to validate.

For file imports, use temporary directories and synthetic workbooks. Test a valid source, malformed file, permission error where practical, missing columns, and a destination that already exists. Avoid relying on the developer’s Downloads folder or a local database full of unknown data.

### Worked example 2: Use a temporary CSV fixture

Temporary fixtures isolate tests from the course sample file and the machine’s filesystem. Uncomment the pytest example in a test module with `import pytest` to execute it.

In [ ]:
from pathlib import Path
import pandas as pd

def load_sales_csv(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {"order_id", "region", "revenue"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    return frame

# In pytest, use tmp_path to create a source file for each test:
# def test_missing_column_is_reported(tmp_path):
#     path = tmp_path / "bad.csv"
#     path.write_text("order_id,region\n1,North\n", encoding="utf-8")
#     with pytest.raises(ValueError, match="revenue"):
#         load_sales_csv(path)

## 3. GUI testing strategy

Automated end-to-end tests for native GUI applications can be platform-sensitive, so start with testable construction and service boundaries. Keep widget creation separate from `mainloop`, expose methods such as `render_records(frame)` and `set_status(message)`, and verify that the view receives the expected values. Add a small smoke test that creates and destroys a root only where a display is available.

Manual test scripts are valuable if repeatable. List the actions, expected visible state, sample file, and platform/version. For critical workflows, test the full path from source selection through normalization, filtering, table rendering, chart update, and export. Include regression examples for known defects rather than trusting visual spot checks alone.

## Guided lab: Lab: regression test suite

Create at least 15 automated tests covering metrics, filters, schema validation, date boundaries, import failures, sorting, and export structure. Add a fake repository that simulates an outage and assert that the controller displays an error without discarding prior data. Add one integration test for each database backend only when test instances are configured.

## Review questions and practice

1. Why should service functions be tested independently from widgets?
2. When is a fake repository insufficient?
3. Why are temporary directories useful in file tests?
4. What should a smoke test prove about a GUI?
5. Why are explicit business-rule tests more valuable than only screenshot comparisons?

## Further reading

- pytest documentation: https://docs.pytest.org/en/stable/
- unittest.mock: https://docs.python.org/3/library/unittest.mock.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.